# PHASE 4: Feature Engineering (Factorial Experiment)
**Traceability**
- Issue ID: #4 Feature Engineering & Labeling

## 1. Objectives
- Implement the full factorial experiment pipeline (120 datasets).
- Factors: 
    - **A**: Outlier Handling (8 levels)
    - **B**: Feature Engineering (5 levels)
    - **C**: Dimensionality Reduction (3 levels)
- Output: 120 pairs of (train, test) CSVs in `../data/experiments/`.

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')

# ── Config ───────────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
EXPERIMENTS_DIR = Path('../data/experiments')
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

# ── Load Data ────────────────────────────────────────────────────────────
print("⏳ Loading datasets...")
df_train_raw = pd.read_csv(PROCESSED_DIR / 'cleaned_dataset.csv')
df_test_raw = pd.read_csv(PROCESSED_DIR / 'cleaned_test_dataset.csv')

# Identify sensor columns
SENSOR_COLS = [c for c in df_train_raw.columns if c.startswith('s_')]
INDEX_COLS = ['unit_number', 'time_cycles']
RUL_COL = 'RUL'

# Ensure sorted
df_train_raw.sort_values(INDEX_COLS, inplace=True)
df_test_raw.sort_values(INDEX_COLS, inplace=True)

print(f"✅ Data Loaded. Train: {df_train_raw.shape}, Test: {df_test_raw.shape}")

## 2. Helper Functions (Factors A, B, C)

In [ ]:
def apply_outlier_handling(df_train, df_test, method, cols):
    """Factor A: Outlier Handling"""
    # Make copies to avoid side effects
    tr = df_train.copy()
    te = df_test.copy()
    
    if method == 'A1': # Raw
        pass
        
    elif method == 'A2': # Clipping 1-99
        # Fit on train
        lower = tr[cols].quantile(0.01)
        upper = tr[cols].quantile(0.99)
        # Transform train & test
        tr[cols] = tr[cols].clip(lower=lower, upper=upper, axis=1)
        te[cols] = te[cols].clip(lower=lower, upper=upper, axis=1)
        
    elif method == 'A3': # Fixed Capping (Mean +/- 3 Std)
        means = tr[cols].mean()
        stds = tr[cols].std()
        lower = means - 3 * stds
        upper = means + 3 * stds
        tr[cols] = tr[cols].clip(lower=lower, upper=upper, axis=1)
        te[cols] = te[cols].clip(lower=lower, upper=upper, axis=1)
        
    elif method == 'A4': # Winsorizing (5-95 to be distinct from A2)
        lower = tr[cols].quantile(0.05)
        upper = tr[cols].quantile(0.95)
        tr[cols] = tr[cols].clip(lower=lower, upper=upper, axis=1)
        te[cols] = te[cols].clip(lower=lower, upper=upper, axis=1)
        
    elif method == 'A5': # Robust Scaling
        scaler = RobustScaler()
        tr[cols] = scaler.fit_transform(tr[cols])
        te[cols] = scaler.transform(te[cols])
        
    elif method == 'A6': # MinMax Scaling
        scaler = MinMaxScaler()
        tr[cols] = scaler.fit_transform(tr[cols])
        te[cols] = scaler.transform(te[cols])
        
    elif method == 'A7': # Capping (A3) + MinMax
        # Cap first
        means = tr[cols].mean()
        stds = tr[cols].std()
        lower = means - 3 * stds
        upper = means + 3 * stds
        tr[cols] = tr[cols].clip(lower=lower, upper=upper, axis=1)
        te[cols] = te[cols].clip(lower=lower, upper=upper, axis=1)
        # Then Scale
        scaler = MinMaxScaler()
        tr[cols] = scaler.fit_transform(tr[cols])
        te[cols] = scaler.transform(te[cols])
        
    elif method == 'A8': # Clipping (A2) + MinMax
        # Clip first
        lower = tr[cols].quantile(0.01)
        upper = tr[cols].quantile(0.99)
        tr[cols] = tr[cols].clip(lower=lower, upper=upper, axis=1)
        te[cols] = te[cols].clip(lower=lower, upper=upper, axis=1)
        # Then Scale
        scaler = MinMaxScaler()
        tr[cols] = scaler.fit_transform(tr[cols])
        te[cols] = scaler.transform(te[cols])
        
    return tr, te

def apply_feature_engineering(df, method, cols):
    """Factor B: Feature Engineering"""
    # We assume df is already processed by Factor A
    # But we must group by unit_number
    
    if method == 'B1': # Raw only
        return df
    
    # For B2-B5, we generate features. 
    # We operate on a copy.
    out_df = df.copy()
    grouped = out_df.groupby('unit_number')
    
    # Lags (B2+)
    if method in ['B2', 'B3', 'B4', 'B5']:
        lags = [1, 2, 3, 4, 5]
        for col in cols:
            for lag in lags:
                out_df[f'{col}_lag{lag}'] = grouped[col].shift(lag)
                
    # Rolling (B3+)
    if method in ['B3', 'B4', 'B5']:
        windows = [5, 10]
        for col in cols:
            for w in windows:
                out_df[f'{col}_roll_mean_{w}'] = grouped[col].transform(lambda x: x.rolling(w).mean())
                out_df[f'{col}_roll_std_{w}'] = grouped[col].transform(lambda x: x.rolling(w).std())
                
    # Diff (B4+)
    if method in ['B4', 'B5']:
        for col in cols:
            out_df[f'{col}_diff'] = grouped[col].diff()
            
    # EWMA (B5)
    if method == 'B5':
        spans = [5, 10]
        for col in cols:
            for s in spans:
                out_df[f'{col}_ewma_{s}'] = grouped[col].transform(lambda x: x.ewm(span=s).mean())
                
    # Drop NaNs generated by shifting/rolling
    out_df.dropna(inplace=True)
    
    return out_df

def apply_pca(df_train, df_test, method, feature_cols):
    """Factor C: Dimensionality Reduction"""
    if method == 'C1': # None
        return df_train, df_test
    
    # For C2/C3, we must scale first (StandardScaler is standard for PCA)
    # We apply this scaling on the *feature set* created in Step 3.
    scaler = StandardScaler()
    X_train = scaler.fit_transform(df_train[feature_cols])
    X_test = scaler.transform(df_test[feature_cols])
    
    n_components = 0.95 if method == 'C2' else 0.90
    pca = PCA(n_components=n_components)
    
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    
    # Reconstruct DataFrames
    # We keep non-feature cols (unit, cycle, RUL)
    meta_cols = ['unit_number', 'time_cycles', 'RUL']
    # Note: RUL might be in df_train but not explicitly handled if we dropped rows.
    # We assume df_train has the index aligned.
    
    # Create column names for PCA
    pca_cols = [f'pca_{i}' for i in range(X_train_pca.shape[1])]
    
    tr_pca = pd.DataFrame(X_train_pca, columns=pca_cols, index=df_train.index)
    te_pca = pd.DataFrame(X_test_pca, columns=pca_cols, index=df_test.index)
    
    # Join back metadata
    tr_final = pd.concat([df_train[meta_cols], tr_pca], axis=1)
    te_final = pd.concat([df_test[meta_cols], te_pca], axis=1)
    
    return tr_final, te_final

## 3. Main Experiment Loop
Iterate through 8x5x3 = 120 combinations.

In [ ]:
OUTLIER_METHODS = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8']
FEATURE_METHODS = ['B1', 'B2', 'B3', 'B4', 'B5']
PCA_METHODS = ['C1', 'C2', 'C3']

print("🚀 Starting Factorial Experiment Generation...")
count = 0

# Pre-calculate RUL Clipping (Fixed at 125 for all experiments as per general practice, or raw?)
# Plan says: "RUL (target) - unchanged". 
# Usually we clip RUL to 125 for training stability. Let's do it once here.
# Note: The plan says "RUL (unchanged)". I will stick to Raw RUL or Clipped? 
# Standard practice is Clipped. I will Clip to 125 in the base loaded data.
df_train_raw['RUL'] = df_train_raw['RUL'].clip(upper=125)
df_test_raw['RUL'] = df_test_raw['RUL'].clip(upper=125)

for a in OUTLIER_METHODS:
    # 1. Apply Outlier Handling
    # Note: We pass fresh copies
    df_tr_a, df_te_a = apply_outlier_handling(df_train_raw, df_test_raw, a, SENSOR_COLS)
    
    for b in FEATURE_METHODS:
        # 2. Apply Feature Engineering
        # Note: Features are generated on the sensor cols processed by A
        df_tr_b = apply_feature_engineering(df_tr_a, b, SENSOR_COLS)
        df_te_b = apply_feature_engineering(df_te_a, b, SENSOR_COLS)
        
        # Identify the new feature columns (exclude meta)
        meta_cols = ['unit_number', 'time_cycles', 'RUL']
        # Also exclude setting columns if they exist? 
        # Our cleaned dataset only has sensors + meta.
        feature_cols = [c for c in df_tr_b.columns if c not in meta_cols]
        
        for c in PCA_METHODS:
            # 3. Apply PCA (and Scaling if needed)
            df_tr_final, df_te_final = apply_pca(df_tr_b, df_te_b, c, feature_cols)
            
            # 4. Save
            exp_id = f"EXP_{a}_{b}_{c}"
            train_fname = EXPERIMENTS_DIR / f"{exp_id}_train.csv"
            test_fname = EXPERIMENTS_DIR / f"{exp_id}_test.csv"
            
            df_tr_final.to_csv(train_fname, index=False)
            df_te_final.to_csv(test_fname, index=False)
            
            count += 1
            if count % 10 == 0:
                print(f"... Generated {count}/120 experiments ({exp_id})")

print(f"\n✅ Completed. 120 dataset pairs saved to {EXPERIMENTS_DIR}")